In [ ]:
import os
import shutil
from pathlib import Path

import pandas as pd


In [3]:
def anonymize(dataset: str) -> list:
    
    df = pd.read_csv(dataset)
    nb_lines = df.shape[0]
    
    # histo from desease
    
    df["NAME"] = ""
    df["NAME"] = [f"{i:04d}.png" for i in range(nb_lines)]
    
    df.to_csv(dataset, index=False)

In [16]:
def create_dataset(name: str, output_path: str, output_path_csv: str, quantity: float = 1.0) -> pd.DataFrame:
    """
        params:
        - Quantity: Quantity (in normalize pourcentage) of the dataset filename (input file)
        takes to create the real dataset.
        Create a csv of the subdataset if the quantity < 1
    """
    
    assert quantity > 0
    assert len(output_path) > 0 and len(name) > 0
    
    filename = Path(name)
    output = Path(output_path)
    
    df: pd.DataFrame = pd.read_csv(filename)
    n_rows = 0
    
    if quantity <= 1:
        n_rows = int(len(df) * quantity)
        n_rows += n_rows % 2
    else:
        n_rows = int(quantity)
    
    df = df.iloc[:n_rows]
    
    left_path =  output / "left"
    right_path = output / "right"
    
    left_path.mkdir(parents=True, exist_ok=True)
    right_path.mkdir(parents=True, exist_ok=True)
    
    left_df: pd.DataFrame = df.loc[df["SIDE"] == "L"]
    right_df: pd.DataFrame = df.loc[df["SIDE"] == "R"]
    
    def _create_class(class_df: pd.DataFrame, path: Path):
        for _, row in class_df.iterrows():
        
            source = row["SOURCE"]
            name = row["NAME"]
            
            source = source.strip('"')
            name = name.strip()

            shutil.copy(source, path / name)
        
        
    _create_class(left_df, left_path)
    _create_class(right_df, right_path)
    
    if quantity != 1:
        basename = Path(name).stem
        
        output_csv = Path(output_path_csv, f"{basename}_{quantity}.csv")
        
        df.to_csv(output_csv)
    
    return df

In [12]:
anonymize("source/csv/sources_kfold_train.csv")
create_dataset("source/csv/sources_kfold_train.csv", "train")

anonymize("source/csv/sources_kfold_test.csv")
create_dataset("source/csv/sources_kfold_test.csv", "test")

## Create multiple dataset for benchmark

In [17]:
create_dataset("source/csv/sources_kfold_train.csv", "benchmark/kfold_0.2", "source/csv", quantity=0.2)
create_dataset("source/csv/sources_kfold_train.csv", "benchmark/kfold_0.4", "source/csv", quantity=0.4)
create_dataset("source/csv/sources_kfold_train.csv", "benchmark/kfold_0.6", "source/csv", quantity=0.6)
create_dataset("source/csv/sources_kfold_train.csv", "benchmark/kfold_0.8", "source/csv", quantity=0.8)

,SOURCE,PATIENT,TYPE,SIDE,PATHO,NAME
0,"""Z:\260701\260701_LEL2266859_L_2\260701_LEL226...",LEL,retine,L,NaN,0000.png
1,"""Z:\260701\260701_LEL2266859_R_2\260701_LEL226...",LEL,retine,R,NaN,0001.png
2,"""Z:\260325\260325_ZEN1418552_L\260325_ZEN14185...",ZEN,retine,L,NaN,0002.png
3,"""Z:\260325\260325_ZEN1418552_R\260325_ZEN14185...",ZEN,retine,R,NaN,0003.png
4,"""Z:\260528\260528_CHG_L_1\260528_CHG_L_1_HD\pn...",CHG,retine,L,dlma-wet,0004.png
...,...,...,...,...,...,...
75,"""Z:\260506\260506_CHE2265897_R_1\260506_CHE226...",CHE,retine,R,NaN,0075.png
76,"""Z:\260422\260422_FOB1839734_L_1\260422_FOB183...",FOB,retine,L,NaN,0076.png
77,"""Z:\260422\260422_FOB1839734_R_1\260422_FOB183...",FOB,retine,R,NaN,0077.png
78,"""Z:\260422\260422_GEPT0_L_1\260422_GEPT0_L_1_H...",GEPT,retine,L,NaN,0078.png
